# 05 風の収束量 — 計算のしかたと、機械学習で効くか

## このノートの目的
- **風向・風速**から、風の**東西成分・南北成分**（ベクトル）を作るしくみを知る。
- 風が「**集まる（収束）**」「**散らばる（発散）**」とは何かを、図と式で理解する。
- 収束量の**3つの計算方法**（2地点法・三角形法・最小二乗法）を、自分の手で動かして比べる。
- 観測の**粗さ**（風向が16方位）が、収束量の精度にどう効くかを、シミュレーションで確かめる。
- 収束量を**説明変数に加えると、予測が良くなるか**を、実際のデータで試す。

## なぜ収束量なのか
雨雲（積乱雲）は、**地表付近で風が集まる（収束する）場所**で、空気が上に押し上げられて発達しやすくなります。
午前中に「風が集まりやすい状況」が観測できれば、午後の大雨の手がかりになるかもしれません。

## キーワード
| 用語 | やさしい説明 |
|---|---|
| **風ベクトル（u, v）** | 風を「東向きの成分 u」と「北向きの成分 v」に分けたもの。単位は m/s。 |
| **発散（divergence）** | 風が**広がっていく**度合い。式は **∂u/∂x ＋ ∂v/∂y**（東へ行くほど東風が強くなる、北へ行くほど北風が強くなる、なら広がる）。 |
| **収束（convergence）** | **発散にマイナスをつけたもの**。風が**集まる**度合い。正なら集まっている。 |
| **渦度（vorticity）** | 風が**回っている**度合い。発散とは別の量。 |
| **空間勾配** | 場所によって値がどれだけ変わるか（例: 1km東へ行くと風が何m/s変わるか）。 |
| **量子化** | 連続した値を、とびとびの値に丸めること。風向は **16方位（22.5°刻み）**に丸められている。 |

In [ ]:
# 準備: 必要なライブラリを読み込み、src フォルダの部品（説明変数を作る dataset.py など）を使えるようにします
import sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path("..").resolve() / "src"))   # ノートブックは notebooks/ にあるので、1つ上の src を探す
plt.rcParams["font.family"] = ["Hiragino Sans", "Arial Unicode MS", "sans-serif"]  # 日本語を表示するための設定
plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.width", 140)
import dataset as ds   # 説明変数と正解を作る部品（src/dataset.py）

## 1. 風向・風速を、風ベクトル（u, v）に直す

気象庁の風向は、「**風が吹いてくる向き**」を16方位で表します。**北風は、北から吹いてきて、南へ向かう**風です。
ここが混乱しやすいので、表で確認しましょう。**東向きを +u、北向きを +v** とします。

In [ ]:
# 目的: 8方位の風（風速 5m/s）を、ベクトル (u, v) に直した表と矢印の図を作る
dirs = ["北", "北東", "東", "南東", "南", "南西", "西", "北西"]
ang = np.deg2rad([ds._DIR[d] for d in dirs])            # 風向（吹いてくる向き）を角度に。北=0°、東=90°、南=180°
speed = 5.0
u = -speed * np.sin(ang)                                # 東西成分: 風が吹いてくる向きの「反対」へ進むので、マイナスをつける
v = -speed * np.cos(ang)                                # 南北成分
tab = pd.DataFrame({"風向（吹いてくる向き）": dirs, "u（東向き+）": u.round(2) + 0.0, "v（北向き+）": v.round(2) + 0.0})   # +0.0 は「-0.00」と表示されるのを防ぐ
display(tab)
fig, ax = plt.subplots(figsize=(4.5, 4.5))
ax.quiver(np.zeros(8), np.zeros(8), u, v, angles="xy", scale_units="xy", scale=1, color=plt.cm.tab10(range(8)))
for d, uu, vv in zip(dirs, u, v):
    ax.text(uu * 1.15, vv * 1.15, d + "風", ha="center", va="center", fontsize=8)
ax.set_xlim(-8, 8); ax.set_ylim(-8, 8); ax.set_aspect("equal"); ax.axhline(0, color="gray", lw=0.4); ax.axvline(0, color="gray", lw=0.4)
ax.set_title("矢印は風が「向かう」向き"); plt.show()

**出力の見方**: たとえば**南風**（南から吹いてくる風）は、u=0、v=**+5**（北向き）になります。**北東風**は、南西へ向かうので u も v もマイナスです。
式は、風向の角度を θ（北=0°、東=90°）として、**u = −風速 × sin θ、v = −風速 × cos θ** です。マイナスは「吹いてくる向きの反対へ進む」ことを表します。

> 📝 風速が 0.2m/s 以下の「静穏」のときは、風ベクトルを (0, 0) とします（風向が決まらないため）。

## 2. 発散と収束 — 図で理解する

風が**場所によってどう変わるか**で、発散・収束・回転が決まります。人工の4つの風の場を描き、**発散を数値で計算**します。

In [ ]:
# 目的: 4種類の風の場（収束・発散・回転・シア）を矢印で描き、発散 ∂u/∂x + ∂v/∂y を数値微分で確かめる
g = np.linspace(-2, 2, 9)
X, Y = np.meshgrid(g, g)
fields = {"収束（中心へ集まる）": (-X, -Y), "発散（中心から広がる）": (X, Y), "回転（反時計回り）": (-Y, X), "シア（高さで風が変わる）": (Y, 0 * Y)}
fig, axes = plt.subplots(1, 4, figsize=(14, 3.6))
for ax, (name, (U, V)) in zip(axes, fields.items()):
    dudx = np.gradient(U, g, axis=1)           # ∂u/∂x: 東へ行くと u がどう変わるか
    dvdy = np.gradient(V, g, axis=0)           # ∂v/∂y: 北へ行くと v がどう変わるか
    div = (dudx + dvdy).mean()
    ax.quiver(X, Y, U, V, color="#4c72b0"); ax.set_title(f"{name}\n発散 = {div:+.1f}"); ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

**出力の見方**
- **収束**（左端）: 矢印がすべて中心を向きます。発散は **−2**（負）。**収束量 ＝ −発散 ＝ +2**（正）。
- **発散**: 矢印が中心から外へ向かいます。発散は **+2**。
- **回転・シア**: 矢印は動いていますが、**集まりも広がりもしない**ので、発散は **0** です（回転は「渦度」で測る量です）。
- 発散は、**「東へ行くほど u が増える」(∂u/∂x > 0) か「北へ行くほど v が増える」(∂v/∂y > 0)** で決まります。**1点の風だけでは、場所による変化が分からず、発散は求められません**。**複数の地点が必要**です。

## 3. 2地点法 — いちばん簡単な近似

観測地点が**2つ**しかないとき、**2地点を結ぶ線の方向**の変化だけは分かります。
那覇（南）と名護（北）を結ぶ線に沿って、「**線方向の風**」を比べます。

> 収束量（線方向）＝ −（名護の線方向の風 − 那覇の線方向の風）÷ 2地点の距離

名護（北端）の南向きの風が強く、那覇（南端）の北向きの風が強ければ、**線の中で風が集まっている**ことになります。

In [ ]:
# 目的: 那覇と名護の位置関係と、線方向の単位ベクトルを確認し、ある1時間の2地点法の計算を手で追う
print(f"那覇→名護の距離: {ds._L:.1f} km、線方向の単位ベクトル: 東成分 {ds._EX:.2f}, 北成分 {ds._EY:.2f}（北北東向き）")
ct = ds.conv_table()                                                                # 全時間の2地点法の収束量（単位 1e-6 /s）
h = pd.read_csv("../data/hourly.csv", parse_dates=["time"])
h["hour"] = h.time.dt.hour.replace(0, 24); h["date"] = (h.time - pd.Timedelta(minutes=1)).dt.normalize()
day = pd.Timestamp("2019-09-15"); hr = 10                                           # ← 演習: 日付・時刻を変えてみよう
rows = []
for st in ("naha", "nago"):
    r = h[(h.station == st) & (h.date == day) & (h.hour == hr)].iloc[0]
    uv = ds._uv(h[(h.station == st) & (h.date == day) & (h.hour == hr)].set_index(["date", "hour"])).iloc[0]
    s_comp = uv.u * ds._EX + uv.v * ds._EY                                          # 線方向の成分（内積）
    rows.append({"地点": st, "風向": r.wdir, "風速": r.wind, "u": round(uv.u, 2), "v": round(uv.v, 2), "線方向の成分": round(s_comp, 2)})
display(pd.DataFrame(rows))
c = ct.loc[(day, hr), "conv"]
print(f"2地点法の収束量 = −(名護 − 那覇) ÷ {ds._L:.1f}km × 1000 = {c:+.1f}（単位 1e-6 /s。正なら収束）")

**出力の見方**: 表の「線方向の成分」が、各地点の風を**線の向き（那覇→名護）に射影**した値です（正は北北東向き）。
名護の値から那覇の値を引き、距離で割って、符号を反転すると**収束量**になります。

> ⚠️ これは発散の**一部（線方向だけ）**です。線に**直交する方向**の広がりは、2地点では分かりません。

## 4. 三角形法 — 3地点で、発散を「ちゃんと」求める

観測地点が**3つ**（三角形）あれば、**平面の式**で風の場所による変化を決められます。風がその範囲で**直線的に**変わると仮定します。

```
u(x, y) = u0 + a·x + b·y          （東西成分）
v(x, y) = v0 + c·x + d·y          （南北成分）
```

3地点の位置 (xᵢ, yᵢ) と風 (uᵢ, vᵢ) が分かれば、未知数 u0, a, b（それぞれ3つ）は、**3本の連立方程式**で決まります。
そして、**発散 ＝ ∂u/∂x ＋ ∂v/∂y ＝ a ＋ d**、渦度 ＝ c − b です。

まず、**答えが分かっている人工の風**で、計算が合っているか確かめます。

In [ ]:
# 目的: 三角形法の計算を、3本の連立方程式として自分で解き、ds.divergence_fit と一致することを確かめる
sts = ds.TRIANGLES["tri_c"]                                      # 名護・宮城島・那覇
xy = np.array([ds._xy(s) for s in sts])                         # 3地点の位置（km）。東向き x、北向き y
print("3地点の位置（km）:\n", pd.DataFrame(xy, index=sts, columns=["x（東）", "y（北）"]).round(1))

# 人工の風: 中心へ向かって吹く（収束）。u = -k·x, v = -k·y。k = 0.01 (m/s)/km
k = 0.01
u, v = -k * xy[:, 0], -k * xy[:, 1]
A = np.column_stack([np.ones(3), xy[:, 0], xy[:, 1]])           # 各行: [1, x_i, y_i]
cu = np.linalg.solve(A, u)                                       # u の式の係数 [u0, a, b]
cv = np.linalg.solve(A, v)                                       # v の式の係数 [v0, c, d]
div = cu[1] + cv[2]                                              # 発散 = a + d
print(f"\n手で解いた発散: {div * 1e3:+.1f}（1e-6 /s）  ／ 理論値 −2k = {-2 * k * 1e3:+.1f}")
print(f"ds.divergence_fit: {ds.divergence_fit(xy, u, v)[0]:+.1f}")

**出力の見方**: 手で解いた発散と、部品 `divergence_fit` の結果が、理論値の **−20**（収束量なら **+20**）と一致します。計算は正しく動いています。

### 地図で見る — どの3地点を使うか
沖縄本島で風を観測しているアメダスは**6地点**です。3地点ずつ**三角形**を作ります。

In [ ]:
# 目的: 6地点の位置（東西 x・南北 y、km）と、使う三角形を地図のように描く
fig, ax = plt.subplots(figsize=(5.5, 6.5))
names = {"oku": "奥", "nago": "名護", "miyagijima": "宮城島", "naha": "那覇", "ashimine": "安次嶺", "itokazu": "糸数"}
for st in ds.LSQ_STATIONS:
    px, py = ds._xy(st); ax.plot(px, py, "ko"); ax.text(px + 1, py + 1, names[st], fontsize=10)
cols = {"tri_n": "#4c72b0", "tri_c": "#dd8452", "tri_s": "#55a868"}
for name, sts in ds.TRIANGLES.items():
    pts = np.array([ds._xy(s) for s in sts]); pts = np.vstack([pts, pts[:1]])
    ax.plot(pts[:, 0], pts[:, 1], color=cols[name], lw=2, label=f"{name}: " + "・".join(names[s] for s in sts))
ax.set_aspect("equal"); ax.set_xlabel("東へ（km）"); ax.set_ylabel("北へ（km）"); ax.legend(fontsize=8, loc="lower left"); plt.show()

**出力の見方**: 色のついた3つの三角形が、三角形法で使う組み合わせです（`tri_n`＝北部〜中部、`tri_c`＝中部〜那覇、`tri_s`＝中部〜南部）。
**安次嶺と那覇は5kmしか離れていない**ので、一緒に三角形にすると**細長く**なり、計算が不安定になります。そのため、三角形には那覇だけを使い、安次嶺は次の「最小二乗法」で使います。

## 5. 最小二乗法 — 地点が3つより多いとき

6地点あるなら、**全部を使ってほうが良い**はずです。3地点なら式がちょうど解けますが、4地点以上だと、**全部には完全に合う式がありません**。
そこで、**誤差の二乗和が最小**になる平面を探します（**最小二乗法**）。1地点の観測がぶれていても、平均化されて影響が小さくなります。

> ✔ `divergence_fit` は、地点が3つなら**厳密な解（三角形法）**、4つ以上なら**最小二乗**を返す、同じ関数です。

In [ ]:
# 目的: 全時間の三角形法・最小二乗法の収束量を計算し、2地点法と並べて見る（風の追加データが必要）
import os
HAVE = os.path.exists("../data/wind_hourly.csv")
if not HAVE:
    print("data/wind_hourly.csv がありません。`python src/fetch_wind.py` で取得してください（約2時間）。")
else:
    tt = ds._tri_cached()                                       # 少し時間がかかる（数十秒）。同じ時間の収束量を4通りの方法で
    allc = pd.concat([ct["conv"].rename("2地点法"), tt], axis=1).dropna()
    show = allc.loc[(day, slice(6, 12)), :].round(1)
    show.columns = ["2地点法", "三角形 北", "三角形 中", "三角形 南", "最小二乗(6地点)"]
    display(show)
    print("方法どうしの相関（全時間）:"); display(allc.corr().round(2).rename(columns={"conv": "2地点法"}, index={"conv": "2地点法"}))

**出力の見方**: 同じ時刻でも、方法によって**収束量がかなり違う**ことがあります。相関の表を見ると、**2地点法は、三角形法・最小二乗法とほぼ無相関**（−0.15〜0.30）です。三角形法どうし・最小二乗法とは、0.24〜0.84と、場所が近いほど高くなります。
「風の収束」といっても、**どこを測るか**で値が違うのは、観測地点が少なく、**風が場所によって複雑に変わる**ためです。

## 6. 観測の粗さが、収束量をどれだけ壊すか — シミュレーション

アメダスの風向は**16方位（22.5°刻み）**、風速は**0.1m/s刻み**です。この**丸め（量子化）**が、収束量の誤差をどれだけ生むのかを、**真の値が分かっている人工の風**で調べます。

In [ ]:
# 目的: 真の収束量が分かっている人工の風を作り、観測と同じ粗さ（16方位・0.1m/s）に丸めて、各方法の推定誤差を見る
rng = np.random.default_rng(0)
DIRS = np.array(list(ds._DIR.values()))                                   # 16方位の角度
def quantize(u, v):
    sp = np.hypot(u, v); th = (np.degrees(np.arctan2(-u, -v)) % 360)       # 風が吹いてくる向き（北=0°）
    th = DIRS[np.argmin(np.abs(((th[:, None] - DIRS[None, :]) + 180) % 360 - 180), axis=1)]   # 一番近い16方位に丸める
    sp = np.round(sp, 1)                                                   # 風速は0.1m/s刻み
    a = np.deg2rad(th); return -sp * np.sin(a), -sp * np.cos(a)

pos = {s: np.array(ds._xy(s)) for s in ds.LSQ_STATIONS}
xs = np.array([pos[s] for s in ds.LSQ_STATIONS]); ix = {s: j for j, s in enumerate(ds.LSQ_STATIONS)}
idx3 = [ix[s] for s in ds.TRIANGLES["tri_c"]]

def simulate(sigma_local, n=2000, quant=True):
    # sigma_local: 地点ごとの局地的な風の違い（m/s）。0なら、近い地点は同じ風向に丸められて、丸め誤差が打ち消し合う（楽観的）
    res = {"2地点法（那覇・名護）": [], "三角形法（名護・宮城島・那覇）": [], "最小二乗法（6地点）": []}; truth = []
    for _ in range(n):
        c_true = rng.normal(0, 40)                                          # 真の収束量（1e-6 /s）。実データの標準偏差は約40
        ub, vb = rng.uniform(-6, 6, 2)                                      # 全体の風（一様な風。収束には寄与しない）
        kk = -c_true * 1e-3 / 2                                             # 等方的な収束: a = d = −c/2（(m/s)/km）
        u = ub + kk * xs[:, 0] + rng.normal(0, sigma_local, 6)              # 風 = 一様な風 + 収束の成分 + 地点ごとの局地的な違い
        v = vb + kk * xs[:, 1] + rng.normal(0, sigma_local, 6)
        if quant:
            u, v = quantize(u, v)                                           # 観測の粗さを再現
        sN = u[ix["nago"]] * ds._EX + v[ix["nago"]] * ds._EY; sS = u[ix["naha"]] * ds._EX + v[ix["naha"]] * ds._EY
        res["2地点法（那覇・名護）"].append(-(sN - sS) / ds._L * 1e3)
        res["三角形法（名護・宮城島・那覇）"].append(-ds.divergence_fit(xs[idx3], u[idx3], v[idx3])[0])
        res["最小二乗法（6地点）"].append(-ds.divergence_fit(xs, u, v)[0])
        truth.append(c_true)
    truth = np.array(truth); rows = {}
    for k, vv in res.items():
        vv = np.array(vv); slope = (vv @ truth) / (truth @ truth)           # 推定 ≒ 傾き × 真の値（傾きが1なら、大きさも合っている）
        noise = np.std(vv - slope * truth)                                  # 比例を補正したあとに残るノイズ
        rows[k] = {"傾き（1が理想）": slope, "ノイズの標準偏差": noise, "S/N比（信号÷ノイズ）": slope * truth.std() / noise, "真の値との相関": np.corrcoef(vv, truth)[0, 1]}
    return pd.DataFrame(rows).T

print("真の収束量の標準偏差: 約40（1e-6 /s）")
for sg in (0.0, 1.0):
    print(f"\n■ 地点ごとの局地的な風の違い σ = {sg} m/s（{'ない（楽観的）' if sg == 0 else '1m/s 程度ある'}）")
    display(simulate(sg).round(2))

**出力の見方**
- **傾き**: 推定値が、真の値の**何倍**で出ているか。**2地点法は約0.5**です。線方向しか測れないので、等方的な収束の**半分**しか拾えません。三角形法・最小二乗法は約1.0（大きさも合っている）。
- **S/N比**（信号÷ノイズ）: **1より小さいと、ノイズのほうが信号より大きい**ことを意味します。
- **σ = 0（局地的な違いがない）**: S/N は 1.5〜1.9 で、丸めだけなら、ある程度は測れます。ただし、近い地点が**同じ16方位に丸められて、丸め誤差が打ち消し合う**ので、楽観的な結果です。
- **σ = 1 m/s（地点ごとに風が1m/sほど違う）**: S/N は **0.4〜0.7** に落ち、**ノイズが信号を上回ります**。実際の沖縄本島は、海岸線や地形のため、近い地点でも風はこのくらい違います（那覇と名護の風速差の標準偏差は約1.7m/s）。
- **方法による違い**: 最小二乗法（6地点）は三角形法（3地点）より S/N が高い＝**地点を増やすと平均化されて安定**します。2地点法の S/N が高く見えるのは、**2地点の距離が51kmと長い**からです（距離が長いほど、同じ風の差でも勾配の誤差が小さくなる）。

> 📝 **まとめると**: アメダスの風から収束量を作るのは、**原理的には可能**ですが、**風向の丸め**と**局地的な風の違い**、**地点の少なさ**のため、**信号に対してノイズが大きい**。これを機械学習の説明変数にして、効くのでしょうか。

## 7. 機械学習で効くか — 実データで試す

**収束量の特徴を加える／加えない**で、予測の PR-AUC を比べます（2010〜2024年の leave-one-year-out、ExtraTrees）。
比べるのは次の5通りです。

| 名前 | 加える特徴量 |
|---|---|
| なし | （収束量なし。基準） |
| 2地点法 | 那覇・名護の線方向の収束量（6特徴） |
| 三角形法 | 3つの三角形の収束量とその平均（6特徴） |
| 最小二乗法 | 6地点の最小二乗の収束量・平均・変化（4特徴） |
| 全部 | 上のすべて |

In [ ]:
# 目的: 収束量の特徴を足す／足さないで、PR-AUC の差と95%区間を比べる（那覇・名護、地域の正解 × 20mm/h）
from sklearn.metrics import average_precision_score
import run_phase2 as p2
target = "y20_region"
cfg = {**p2.BASE, "min_samples_leaf": 20, "n_estimators": 100, "change": False}
LSQ = ["conv_lsq", "conv_lsq_mean", "conv_lsq_d1", "conv_lsq_d2"]
TRI = [c for c in ds.FEATURES_TRI if c.startswith("conv_tri")]

def bootstrap(y, days, preds, B=1000, seed=0):
    rng = np.random.default_rng(seed); udays = np.unique(days); idx = {d: np.where(days == d)[0] for d in udays}
    out = {k: [] for k in preds}
    for _ in range(B):
        pick = np.concatenate([idx[d] for d in rng.choice(udays, len(udays))])
        if y[pick].sum() == 0: continue
        for k, p in preds.items(): out[k].append(average_precision_score(y[pick], p[pick]))
    return {k: np.array(v) for k, v in out.items()}

if not HAVE:
    print("風の追加データがありません。")
else:
    for st, nm in (("naha", "那覇"), ("nago", "名護")):
        x = ds.build(st); base = ds.features(st, change=False)
        sets = {"なし": base, "2地点法": base + ds.FEATURES_CONV, "三角形法": base + TRI, "最小二乗法": base + LSQ, "全部": base + ds.FEATURES_CONV + TRI + LSQ}
        use = (x.year.isin(range(2010, 2025)) & x.wind_ok).to_numpy()
        xe = x[use].reset_index(drop=True); y = xe[target].to_numpy(); days = xe.date.to_numpy()
        P = {k: np.mean([p2.oof(x, f, target, "et", cfg, s) for s in range(3)], axis=0)[use] for k, f in sets.items()}
        b = bootstrap(y, days, P, B=600)
        rows = []
        for k in P:
            d = b[k] - b["なし"]; lo, hi = np.percentile(d, [2.5, 97.5])
            rows.append({"特徴": k, "PR-AUC": average_precision_score(y, P[k]), "「なし」との差": d.mean(), "差の95%区間": f"{lo:+.3f}〜{hi:+.3f}"})
        print(f"【{nm}】標本 {len(y)}、陽性 {y.sum()}（{xe.loc[y == 1, 'date'].nunique()}日）")
        display(pd.DataFrame(rows).set_index("特徴").round(3))

**出力の見方（この結果の読み方）**: 「差の95%区間」が **0 をまたいでいれば、収束量を加えても良くなったとは言えません**（ノート03）。
### このプロジェクトでの結果（上の表の読み方）
上の表は、**木100本・3シード**の簡易版です。本番（木300本・10シード、収束量の有無の比較）は `outputs/phase6.md`（2地点法）・`outputs/phase6b.md`（三角形法・最小二乗法）にあり、傾向は同じです。

- **どの方法でも、収束量を加えて良くなったとは言えません。** 差の95%区間は、ほぼすべて **0をまたぎます**（差はプラスにもマイナスにも出て、ぶれの範囲内）。
- **那覇の2地点法だけ、少し悪化**しました（本番: 差 −0.007、区間 −0.015〜−0.002）。ノイズの多い特徴を足すと、木が**ノイズに引きずられた**可能性があります。
- **収束量だけをスコアにしても、ほとんど当たりません**（ROC-AUC 0.5〜0.6。0.5は無相関）。最も高いのは、那覇の最小二乗法の約0.60で、「**収束が強いほど降りやすい**」傾向は**弱いながらある**、という程度です。
- **理由の候補**: ①風向が16方位で粗い。②地点が少なく、局地的な風の違い（海岸線・地形）がノイズになる（§6のシミュレーションで、S/Nが1より小さい）。③午後の大雨は、**午前の風の収束だけでは決まらない**。
- つまり、「**物理的に筋が通る特徴量でも、データの粗さ・量が足りないと、効果は検出できない**」という学びです。ここで大事なのは、**効かなかった**ことを、区間つきで確かめた点です。

## まとめ
- 風向・風速は、**u（東西）・v（南北）の成分**に直して使う。**風向は「吹いてくる向き」**なので、符号に注意。
- **発散は ∂u/∂x ＋ ∂v/∂y**。**収束＝−発散**。**複数の地点**がないと求められない。
- **2地点法**は線方向だけ、**三角形法**は3点で厳密、**最小二乗法**は多点で平均化。
- 観測の**粗さ（16方位・0.1m/s）**と**地点の少なさ**で、**収束量はノイズが大きい**。
- 機械学習の説明変数として効くかは、**実際に試して、評価の不確かさ（ノート03）まで含めて判断する**。

## 演習
1. **1.** の `speed = 5.0` を `10.0` に変えてください（観察点: 矢印の長さが変わるだけで、向きは変わらないか）。
2. **2.** の `fields` に `"一様な風": (0*X + 1, 0*Y)` を追加してください（観察点: 発散はいくつになるか。風が一様なら、収束も発散もしない）。
3. **3.** の `day` と `hr` を変えてください（観察点: 収束量の符号は、風向の向きとどう対応するか。名護が北風、那覇が南風なら正になるか）。
4. **6.** の `rng.normal(0, 40)` を `rng.normal(0, 200)`（信号を大きく）に変えてください（観察点: S/N比は大きくなるか。信号が大きいと、粗い観測でも測りやすい）。
5. **6.** の `simulate(sg)` を `simulate(sg, quant=False)` に変えてください（観察点: 風向を16方位に丸めない場合、S/N はどれだけ上がるか。**風向の丸めがどのくらい効いているか**が分かる）。
6. **6.** の `for sg in (0.0, 1.0):` を `(0.0, 0.5, 1.0, 2.0)` に変えてください（観察点: 局地的な風の違いが大きくなるほど、S/N比はどう変わるか）。